In [4]:
import polars as pl
dataset_1 = pl.read_excel('../data/transcribed/object_naming.xlsx')
dataset_2 = pl.read_excel('../data/transcribed/'
                              'object&action_naming.xlsx')


Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 12, falling back to string


In [5]:
columns_for_now = [
    'StimSite',
    'Stimulus',
    'RT_start',
    'Response_annot',
    'Response_transcription_annot',
    'Error_type_annot'
]

dataset = pl.concat([
    dataset_1[columns_for_now],
    dataset_2.rename({'RT_start_annot': 'RT_start'})[columns_for_now]
])


In [6]:
print(len(dataset))

22260


In [7]:
dataset = dataset\
    .with_columns(pl.col("Error_type_annot").str.to_lowercase())


In [ ]:
pl.Config.set_tbl_rows(-1)
print(
*dataset['Error_type_annot']\
    .value_counts()\
    .sort('count', descending=True).rows(),
    sep='\n'
)


In [10]:
dataset = dataset.with_columns(
    pl.col('Error_type_annot').replace({
    ' ': None,
    's/a': 'speech arrest',
    'фонетическа парафазия': 'фонетическая парафазия',
    'фонетическая парфазия': 'фонетическая парафазия',
    'задрежка': 'задержка',
    'задржка': 'задержка',
    'подбор слова': 'поиск слова'
    })
)


In [11]:
dataset[['Error_type_annot']] = dataset[['Error_type_annot']].fill_null('нет')

In [12]:
allowed_types = dataset['Error_type_annot']\
        .value_counts()\
        .sort('count', descending=True)['Error_type_annot'][:8]


In [13]:
dataset_clean = dataset.filter(
    pl.col('Error_type_annot').is_in(allowed_types)
)


C:\Users\Vlad\AppData\Local\Temp\ipykernel_1316\1244736227.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  dataset_clean = dataset.filter(


In [14]:
string_column = dataset_clean[['Error_type_annot']]

dataset_clean_onehot = dataset_clean\
    .to_dummies(columns=['Error_type_annot'])\
    .rename(lambda col: col.replace('Error_type_annot_', ''))

dataset_clean_onehot[['Error_type_annot']] = string_column


In [ ]:
dataset_clean_onehot.head()


In [16]:
error_type_ids = {
    'нет': 0,
    'speech arrest': 1,
    'аномия': 2,
    'дизартрия': 3,
    'задержка': 4,
    'поиск слова': 5,
    'семантическая парафазия': 6,
    'фонетическая парафазия': 7
}


In [17]:
from json import dump

error_type_ids_file = open('../data/processed/error_type_ids.json',
                           'w', -1, 'utf-8')
dump(error_type_ids, error_type_ids_file)
error_type_ids_file.close()


In [18]:
dataset_clean_onehot_label = dataset_clean_onehot.with_columns(
    pl.col('Error_type_annot')\
    .replace(error_type_ids)\
    .cast(pl.Int8)
    .alias('error_type_label')
)


In [ ]:
dataset_clean_onehot_label.sample(10)


In [20]:
dataset_clean_onehot_label.write_csv('../data/processed/'
                               'dataset_clean_onehot_label.csv')


In [21]:
print(len(dataset_clean_onehot_label))


22146
